**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 Session 12: Advanced RAG - HyDE, Reranking, Ensemble Retriever

## 📋 학습 목표

1️⃣ 기본 RAG의 한계점을 이해한다  
2️⃣ HyDE (Hypothetical Document Embeddings)를 구현한다  
3️⃣ Reranking으로 검색 결과 품질을 향상시킨다  
4️⃣ Ensemble Retriever (BM25 + 시맨틱)를 구현한다  
5️⃣ Parent Document Retriever를 이해하고 구현한다  
6️⃣ 각 기법의 성능을 비교한다  

---

### 🖥️ 실습 환경
- **GPU**: 선택사항 (Reranker 사용 시 GPU 권장)
- **필수 패키지**: langchain, chromadb, sentence-transformers, rank_bm25

## 1️⃣ 기본 RAG의 한계

기본 RAG는 단순 유사도 검색에 의존하기 때문에 여러 한계가 있습니다.

### 🔴 기본 RAG의 문제점

| 문제 | 설명 | 해결 기법 |
|------|------|----------|
| 질문-문서 불일치 | 질문 형태와 문서 형태가 다름 | **HyDE** |
| 검색 품질 저하 | Top-k 결과 중 관련 없는 문서 포함 | **Reranking** |
| 키워드 누락 | 의미는 같지만 단어가 다른 경우 | **Ensemble Retriever** |
| 문맥 손실 | 작은 청크로 인한 맥락 부족 | **Parent Document Retriever** |

### 📊 Advanced RAG 기법 체계

```
Advanced RAG
├── Pre-Retrieval (검색 전)
│   ├── HyDE (가상 문서 생성)
│   └── Query Expansion (쿼리 확장)
├── Retrieval (검색)
│   ├── Ensemble Retriever (BM25 + 시맨틱)
│   └── Parent Document Retriever
└── Post-Retrieval (검색 후)
    ├── Reranking (재순위화)
    └── Context Compression (컨텍스트 압축)
```

In [ ]:
# 📦 패키지 확인
import importlib
import os

packages = [
    "langchain",
    "langchain_community",
    "chromadb",
    "sentence_transformers",
    "rank_bm25",
]

print("📦 패키지 버전 확인")
print("=" * 40)
for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "installed")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치 필요")
        print(f"     pip install {pkg_name.replace('_', '-')}")

In [ ]:
# 🔧 GPU 메모리 유틸리티 (GPU 사용 시)
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print(f"[{tag}] CPU 모드로 실행 중")

print_gpu_memory("초기 상태")

In [ ]:
# 📄 실습용 문서 준비
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 한국어 AI 관련 문서
documents = [
    Document(
        page_content="""대규모 언어 모델(LLM)은 수십억 개의 파라미터를 가진 딥러닝 모델입니다.
GPT-4, Claude, Gemini 등이 대표적인 LLM입니다.
이 모델들은 대량의 텍스트 데이터로 사전학습되어 다양한 자연어 처리 작업을 수행할 수 있습니다.
LLM의 핵심 능력은 문맥을 이해하고 자연스러운 텍스트를 생성하는 것입니다.
최근에는 소형 언어 모델(sLLM)이 주목받고 있으며, 1B~7B 크기로도 우수한 성능을 보여줍니다.""",
        metadata={"source": "llm_basics.txt", "topic": "LLM"}
    ),
    Document(
        page_content="""파인튜닝(Fine-tuning)은 사전학습된 모델을 특정 작업에 맞게 추가 학습하는 방법입니다.
Full Fine-Tuning은 모든 파라미터를 업데이트하며, 가장 높은 성능을 달성할 수 있지만 많은 리소스가 필요합니다.
LoRA(Low-Rank Adaptation)는 저랭크 행렬 분해를 통해 학습 파라미터 수를 대폭 줄입니다.
QLoRA는 4bit 양자화와 LoRA를 결합한 방법으로, RTX 4060 같은 소비자급 GPU에서도 7B 모델 학습이 가능합니다.
파인튜닝 시에는 학습률, 배치 크기, 에폭 수 등의 하이퍼파라미터를 적절히 설정해야 합니다.""",
        metadata={"source": "finetuning.txt", "topic": "파인튜닝"}
    ),
    Document(
        page_content="""벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 데이터베이스입니다.
텍스트를 임베딩 벡터로 변환하여 저장하면, 의미 기반 유사도 검색이 가능합니다.
ChromaDB는 오픈소스 벡터 DB로, 간단한 API와 인메모리/영구 저장 모드를 제공합니다.
FAISS는 Meta에서 개발한 초고속 유사도 검색 라이브러리입니다.
Pinecone은 관리형 클라우드 벡터 DB로, 스케일링이 용이합니다.
Weaviate는 GraphQL 기반의 벡터 검색 엔진입니다.""",
        metadata={"source": "vectordb.txt", "topic": "벡터DB"}
    ),
    Document(
        page_content="""프롬프트 엔지니어링은 LLM에서 원하는 출력을 얻기 위한 입력 설계 기술입니다.
제로샷 프롬프팅은 예시 없이 직접 작업을 지시합니다.
퓨샷 프롬프팅은 몇 개의 입출력 예시를 함께 제공합니다.
Chain-of-Thought(CoT)은 단계별 추론 과정을 유도하여 복잡한 문제 해결 능력을 높입니다.
시스템 프롬프트로 AI의 역할, 제약조건, 응답 형식 등을 정의할 수 있습니다.
프롬프트의 품질이 LLM 출력의 품질을 크게 좌우합니다.""",
        metadata={"source": "prompt_eng.txt", "topic": "프롬프트"}
    ),
    Document(
        page_content="""강화학습(Reinforcement Learning)은 에이전트가 환경과 상호작용하며 보상을 최대화하는 학습 방법입니다.
RLHF(Reinforcement Learning from Human Feedback)는 인간의 선호도를 학습에 반영합니다.
PPO(Proximal Policy Optimization)는 안정적인 정책 업데이트를 보장하는 알고리즘입니다.
DPO(Direct Preference Optimization)는 보상 모델 없이 선호도를 직접 학습합니다.
GRPO(Group Relative Policy Optimization)는 DeepSeek에서 제안한 효율적인 강화학습 방법입니다.
이러한 기법들은 LLM의 안전성과 유용성을 향상시키는 데 사용됩니다.""",
        metadata={"source": "rl_methods.txt", "topic": "강화학습"}
    ),
]

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
splits = text_splitter.split_documents(documents)

print(f"📄 문서 준비 완료")
print(f"  원본: {len(documents)}개 → 청크: {len(splits)}개")

In [ ]:
# 🔧 기본 벡터스토어 및 LLM 설정
USE_OLLAMA = True

# 임베딩 모델
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# 벡터스토어
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="advanced_rag_demo",
)

# LLM 설정
if USE_OLLAMA:
    from langchain_community.llms import Ollama
    llm = Ollama(model="qwen2.5:1.5b", temperature=0.3)
    print("✅ Ollama LLM 설정 완료")
else:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    print("✅ OpenAI LLM 설정 완료")

# 기본 Retriever
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✅ 벡터스토어 구축 완료 ({vectorstore._collection.count()}개 청크)")

## 2️⃣ 기본 RAG 베이스라인

개선 효과를 측정하기 위한 기본 RAG의 성능을 확인합니다.

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import time

rag_prompt = PromptTemplate(
    template="""다음 컨텍스트를 참고하여 질문에 한국어로 답변해주세요.

컨텍스트:
{context}

질문: {question}
답변:""",
    input_variables=["context", "question"]
)

# 📊 테스트 질문들
test_questions = [
    "소형 언어 모델(sLLM)이 뭐야?",                    # 간단한 질문
    "GPU 메모리가 적어도 큰 모델을 학습할 수 있는 방법은?",  # 간접적 질문
    "LLM을 더 안전하게 만드는 학습 방법은?",              # 추론 필요
    "벡터 검색에 사용할 수 있는 오픈소스 도구들은?",        # 세부 정보
    "AI 모델의 출력 품질을 높이는 입력 기법은?",           # 간접 표현
]

def evaluate_retriever(retriever, name, questions=test_questions):
    """Retriever의 검색 결과를 평가합니다."""
    print(f"\n📊 [{name}] 검색 결과 평가")
    print("=" * 60)
    
    results = []
    for q in questions:
        start = time.time()
        docs = retriever.invoke(q)
        elapsed = time.time() - start
        
        print(f"\n  ❓ {q}")
        print(f"  ⏱️ {elapsed:.3f}초 | 검색 {len(docs)}개")
        for i, doc in enumerate(docs[:3], 1):
            source = doc.metadata.get('source', 'unknown')
            print(f"    {i}. [{source}] {doc.page_content[:60]}...")
        
        results.append({
            "question": q,
            "docs": docs,
            "elapsed": elapsed,
        })
    
    avg_time = sum(r["elapsed"] for r in results) / len(results)
    print(f"\n  📊 평균 검색 시간: {avg_time:.3f}초")
    return results

# 기본 RAG 베이스라인
base_results = evaluate_retriever(base_retriever, "기본 RAG")

## 3️⃣ HyDE (Hypothetical Document Embeddings)

**HyDE**는 질문을 가상의 답변 문서로 변환한 후 검색하는 기법입니다.

### 🔑 HyDE의 핵심 아이디어
```
기본 RAG:  질문 임베딩 → 문서 임베딩과 비교
HyDE:      질문 → LLM으로 가상 답변 생성 → 가상 답변 임베딩 → 문서 임베딩과 비교
```

- 🔹 질문과 문서의 형태 차이를 해소
- 🔹 "문서 ↔ 문서" 유사도 비교로 변환
- 🔹 LLM의 지식을 검색 쿼리에 활용

In [ ]:
from langchain.chains import LLMChain
from langchain_core.runnables import RunnablePassthrough

# 📝 HyDE 프롬프트 정의
hyde_prompt = PromptTemplate(
    template="""다음 질문에 대한 답변을 작성해주세요.
실제 정보가 아니어도 괜찮으니, 관련 기술 문서처럼 상세하게 작성해주세요.

질문: {question}

답변 문서:""",
    input_variables=["question"]
)

class HyDERetriever:
    """HyDE (Hypothetical Document Embeddings) Retriever"""
    
    def __init__(self, llm, base_retriever, embeddings):
        self.llm = llm
        self.base_retriever = base_retriever
        self.embeddings = embeddings
        self.hyde_prompt = hyde_prompt
    
    def invoke(self, query):
        # 1. LLM으로 가상 문서 생성
        hypothetical_doc = self.llm.invoke(
            self.hyde_prompt.format(question=query)
        )
        if not isinstance(hypothetical_doc, str):
            hypothetical_doc = hypothetical_doc.content
        
        # 2. 가상 문서로 검색
        docs = self.base_retriever.invoke(hypothetical_doc)
        return docs
    
    def get_relevant_documents(self, query):
        return self.invoke(query)

# HyDE Retriever 생성
hyde_retriever = HyDERetriever(
    llm=llm,
    base_retriever=base_retriever,
    embeddings=embeddings,
)

print("✅ HyDE Retriever 생성 완료")
print("  📊 동작: 질문 → LLM(가상 문서 생성) → 벡터 검색")

In [ ]:
# 🔍 HyDE 동작 과정 살펴보기
print("🔍 HyDE 동작 과정 상세")
print("=" * 60)

test_query = "GPU 메모리가 적어도 큰 모델을 학습할 수 있는 방법은?"
print(f"❓ 원래 질문: {test_query}")

# 1단계: 가상 문서 생성
print(f"\n📝 1단계: 가상 문서 생성")
hypothetical = llm.invoke(hyde_prompt.format(question=test_query))
hypo_text = hypothetical if isinstance(hypothetical, str) else hypothetical.content
print(f"   {hypo_text[:200]}...")

# 2단계: 가상 문서로 검색
print(f"\n🔍 2단계: 가상 문서로 벡터 검색")
hyde_docs = base_retriever.invoke(hypo_text)
for i, doc in enumerate(hyde_docs[:3], 1):
    print(f"   {i}. [{doc.metadata.get('source', '')}] {doc.page_content[:80]}...")

# 비교: 원래 질문으로 검색
print(f"\n📊 비교: 원래 질문으로 직접 검색")
base_docs = base_retriever.invoke(test_query)
for i, doc in enumerate(base_docs[:3], 1):
    print(f"   {i}. [{doc.metadata.get('source', '')}] {doc.page_content[:80]}...")

In [ ]:
# 📊 HyDE 평가
hyde_results = evaluate_retriever(hyde_retriever, "HyDE RAG")

## 4️⃣ Reranking (재순위화)

**Reranking**은 초기 검색 결과를 더 정교한 모델로 다시 순위를 매기는 기법입니다.

### 🔑 Reranking의 원리
```
1단계: 빠른 검색 (Bi-encoder) → 후보 문서 N개
2단계: 정밀 평가 (Cross-encoder) → 재순위화 → 상위 K개
```

- 🔹 **Bi-encoder**: 질문과 문서를 독립적으로 인코딩 (빠르지만 덜 정확)
- 🔹 **Cross-encoder**: 질문+문서를 함께 인코딩 (느리지만 더 정확)

In [ ]:
from sentence_transformers import CrossEncoder

# 🔧 Cross-encoder Reranker 로딩
print("🔧 Cross-encoder Reranker 모델 로딩 중...")

reranker_model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512,
)
print("✅ Reranker 모델 로딩 완료")

class RerankingRetriever:
    """Reranking을 적용한 Retriever"""
    
    def __init__(self, base_retriever, reranker, top_k=3, fetch_k=10):
        self.base_retriever = base_retriever
        self.reranker = reranker
        self.top_k = top_k
        self.fetch_k = fetch_k
    
    def invoke(self, query):
        # 1. 초기 검색 (더 많은 후보)
        initial_docs = self.base_retriever.invoke(query)
        
        if not initial_docs:
            return []
        
        # 2. Cross-encoder로 재점수화
        pairs = [[query, doc.page_content] for doc in initial_docs]
        scores = self.reranker.predict(pairs)
        
        # 3. 점수 기준 정렬
        scored_docs = list(zip(initial_docs, scores))
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        
        # 4. 상위 K개 반환
        return [doc for doc, score in scored_docs[:self.top_k]]
    
    def get_relevant_documents(self, query):
        return self.invoke(query)

# 더 많은 후보를 가져오는 Retriever
wide_retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

# Reranking Retriever 생성
reranking_retriever = RerankingRetriever(
    base_retriever=wide_retriever,
    reranker=reranker_model,
    top_k=3,
)

print("✅ Reranking Retriever 생성 완료")
print("  📊 동작: 넓은 검색(8개) → Cross-encoder 재순위화 → 상위 3개")

In [ ]:
# 🔍 Reranking 동작 과정 상세
print("🔍 Reranking 동작 과정 상세")
print("=" * 60)

test_query = "LLM을 더 안전하게 만드는 학습 방법은?"
print(f"❓ 질문: {test_query}")

# 1단계: 넓은 검색
initial_docs = wide_retriever.invoke(test_query)
print(f"\n📋 1단계: 초기 검색 결과 ({len(initial_docs)}개)")

# 2단계: Cross-encoder 점수
pairs = [[test_query, doc.page_content] for doc in initial_docs]
scores = reranker_model.predict(pairs)

print(f"\n📊 2단계: Reranking 점수")
scored = list(zip(initial_docs, scores))
scored.sort(key=lambda x: x[1], reverse=True)

for i, (doc, score) in enumerate(scored, 1):
    marker = "✅" if i <= 3 else "  "
    print(f"  {marker} {i}. (점수: {score:.4f}) [{doc.metadata.get('source', '')}] {doc.page_content[:60]}...")

print(f"\n💡 상위 3개만 최종 결과로 사용")

In [ ]:
# 📊 Reranking 평가
reranking_results = evaluate_retriever(reranking_retriever, "Reranking RAG")

## 5️⃣ Ensemble Retriever (BM25 + 시맨틱)

**Ensemble Retriever**는 키워드 기반 검색(BM25)과 의미 기반 검색(시맨틱)을 결합합니다.

### 🔑 BM25 vs 시맨틱 검색
| 특성 | BM25 | 시맨틱 검색 |
|------|------|------------|
| 매칭 방식 | 키워드 일치 | 의미 유사도 |
| 장점 | 정확한 키워드 매칭 | 동의어/유사 표현 |
| 단점 | 동의어 놓침 | 키워드 정확도 낮음 |
| 속도 | 빠름 | 임베딩 필요 |

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# 🔧 BM25 Retriever 생성
bm25_retriever = BM25Retriever.from_documents(
    splits,
    k=3,
)
print("✅ BM25 Retriever 생성 완료")

# 🔧 Ensemble Retriever 생성 (BM25 + 시맨틱)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, base_retriever],
    weights=[0.4, 0.6],  # BM25 40%, 시맨틱 60%
)

print("✅ Ensemble Retriever 생성 완료")
print("  📊 구성: BM25 (40%) + 시맨틱 검색 (60%)")

In [ ]:
# 🔍 BM25 vs 시맨틱 vs Ensemble 비교
print("🔍 BM25 vs 시맨틱 vs Ensemble 비교")
print("=" * 60)

compare_query = "벡터 검색에 사용할 수 있는 오픈소스 도구들은?"
print(f"❓ 질문: {compare_query}")

# BM25 결과
print(f"\n🔑 BM25 (키워드 기반):")
bm25_docs = bm25_retriever.invoke(compare_query)
for i, doc in enumerate(bm25_docs[:3], 1):
    print(f"  {i}. [{doc.metadata.get('source', '')}] {doc.page_content[:70]}...")

# 시맨틱 결과
print(f"\n🧠 시맨틱 (의미 기반):")
semantic_docs = base_retriever.invoke(compare_query)
for i, doc in enumerate(semantic_docs[:3], 1):
    print(f"  {i}. [{doc.metadata.get('source', '')}] {doc.page_content[:70]}...")

# Ensemble 결과
print(f"\n🔄 Ensemble (BM25 + 시맨틱):")
ensemble_docs = ensemble_retriever.invoke(compare_query)
for i, doc in enumerate(ensemble_docs[:3], 1):
    print(f"  {i}. [{doc.metadata.get('source', '')}] {doc.page_content[:70]}...")

In [ ]:
# 📊 Ensemble 평가
ensemble_results = evaluate_retriever(ensemble_retriever, "Ensemble RAG")

## 6️⃣ Parent Document Retriever

**Parent Document Retriever**는 작은 청크로 검색하되, 큰 문맥(부모 문서)을 반환합니다.

### 🔑 핵심 아이디어
```
문서 → 큰 청크(부모) → 작은 청크(자식)
                        ↓
검색: 작은 청크로 정밀 매칭
반환: 큰 청크(부모)로 풍부한 문맥
```

- 🔹 작은 청크: 검색 정확도 높음
- 🔹 큰 청크: 더 풍부한 문맥 제공
- 🔹 두 가지 장점을 모두 활용

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

# 📏 부모/자식 분할기 설정
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,    # 큰 청크 (LLM에 전달할 컨텍스트)
    chunk_overlap=50,
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,    # 작은 청크 (검색용)
    chunk_overlap=20,
)

# 부모 문서 저장소
docstore = InMemoryStore()

# 새로운 벡터스토어 (Parent Document 용)
parent_vectorstore = Chroma(
    collection_name="parent_doc_demo",
    embedding_function=embeddings,
)

# Parent Document Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=parent_vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가
parent_retriever.add_documents(documents)

print("✅ Parent Document Retriever 생성 완료")
print(f"  📊 부모 청크 크기: 400자")
print(f"  📊 자식 청크 크기: 100자")
print(f"  📊 검색: 자식(100자) → 반환: 부모(400자)")

In [ ]:
# 🔍 Parent Document Retriever 동작 확인
print("🔍 Parent Document Retriever 결과")
print("=" * 60)

test_query = "LoRA의 원리는?"
print(f"❓ 질문: {test_query}")

# 일반 검색 (작은 청크)
print(f"\n🔹 기본 RAG (200자 청크):")
base_docs = base_retriever.invoke(test_query)
for i, doc in enumerate(base_docs[:2], 1):
    print(f"  {i}. ({len(doc.page_content)}자) {doc.page_content[:100]}...")

# Parent Document 검색 (큰 청크 반환)
print(f"\n🔹 Parent Document RAG (100자→400자):")
parent_docs = parent_retriever.invoke(test_query)
for i, doc in enumerate(parent_docs[:2], 1):
    print(f"  {i}. ({len(doc.page_content)}자) {doc.page_content[:150]}...")

print(f"\n💡 Parent Document Retriever는 더 넓은 문맥을 제공합니다!")

In [ ]:
# 📊 Parent Document 평가
parent_results = evaluate_retriever(parent_retriever, "Parent Document RAG")

## 7️⃣ 종합 성능 비교

모든 Advanced RAG 기법의 성능을 종합적으로 비교합니다.

In [ ]:
# 📊 종합 성능 비교
from sentence_transformers import SentenceTransformer
import numpy as np

eval_embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def compute_relevance_score(query, docs):
    """검색 결과의 관련성 점수를 계산합니다."""
    if not docs:
        return 0.0
    q_emb = eval_embedder.encode(query)
    scores = []
    for doc in docs[:3]:
        d_emb = eval_embedder.encode(doc.page_content)
        sim = np.dot(q_emb, d_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(d_emb))
        scores.append(float(sim))
    return np.mean(scores)

# 각 방법의 평균 관련성 점수 계산
retrievers = {
    "기본 RAG": base_retriever,
    "HyDE": hyde_retriever,
    "Reranking": reranking_retriever,
    "Ensemble": ensemble_retriever,
    "Parent Doc": parent_retriever,
}

print("📊 종합 성능 비교")
print("=" * 65)

comparison_results = {}

for name, ret in retrievers.items():
    scores = []
    times = []
    
    for q in test_questions:
        start = time.time()
        try:
            docs = ret.invoke(q)
            elapsed = time.time() - start
            score = compute_relevance_score(q, docs)
            scores.append(score)
            times.append(elapsed)
        except Exception as e:
            times.append(0)
            scores.append(0)
    
    avg_score = np.mean(scores)
    avg_time = np.mean(times)
    comparison_results[name] = {"score": avg_score, "time": avg_time}

# 결과 출력
print(f"\n{'방법':<15} {'평균 관련성':>12} {'평균 시간':>12} {'상대 성능':>12}")
print("-" * 55)

base_score = comparison_results["기본 RAG"]["score"]
for name, result in comparison_results.items():
    relative = (result['score'] / base_score * 100) if base_score > 0 else 0
    bar = "█" * int(result['score'] * 20)
    print(f"{name:<15} {result['score']:.4f}       {result['time']:.3f}초     {relative:.0f}% {bar}")

In [ ]:
# 💡 기법별 적합한 사용 상황 정리
print("💡 Advanced RAG 기법 선택 가이드")
print("=" * 65)

guide = [
    {
        "method": "HyDE",
        "best_for": "질문과 문서 형태가 다를 때",
        "cost": "LLM 호출 추가 (느림)",
        "example": "'GPU 메모리 절약 방법은?' → 기술 문서 검색",
    },
    {
        "method": "Reranking",
        "best_for": "검색 결과에 노이즈가 많을 때",
        "cost": "Cross-encoder 추론 (보통)",
        "example": "비슷한 주제의 문서가 많아 정밀 순위가 필요할 때",
    },
    {
        "method": "Ensemble",
        "best_for": "키워드+의미 검색이 모두 필요할 때",
        "cost": "BM25 인덱스 추가 (빠름)",
        "example": "전문 용어와 일반 표현이 혼합된 도메인",
    },
    {
        "method": "Parent Document",
        "best_for": "더 넓은 문맥이 필요할 때",
        "cost": "저장 공간 추가 (빠름)",
        "example": "연속된 내용의 긴 문서에서 검색할 때",
    },
]

for g in guide:
    print(f"\n🔹 {g['method']}")
    print(f"   적합: {g['best_for']}")
    print(f"   비용: {g['cost']}")
    print(f"   예시: {g['example']}")

print(f"\n📌 실전 팁:")
print(f"  🔹 여러 기법을 조합하여 사용 가능 (예: Ensemble + Reranking)")
print(f"  🔹 도메인과 데이터 특성에 따라 최적 조합이 다름")
print(f"  🔹 반드시 평가(RAGAS 등)로 효과를 검증해야 함")

In [ ]:
# 📌 실습 정리
print("📌 오늘의 핵심 정리")
print("=" * 50)
print("  1️⃣ 기본 RAG의 한계: 질문-문서 불일치, 노이즈, 문맥 부족")
print("  2️⃣ HyDE: LLM으로 가상 문서를 만들어 검색 품질 향상")
print("  3️⃣ Reranking: Cross-encoder로 검색 결과를 정밀 재순위화")
print("  4️⃣ Ensemble: BM25(키워드) + 시맨틱(의미) 결합")
print("  5️⃣ Parent Document: 작은 청크로 검색, 큰 문맥으로 응답")
print("  6️⃣ 기법 선택은 데이터/도메인 특성에 따라 결정")

---

## 🎯 실습 과제

1️⃣ Ensemble Retriever의 weight를 [0.3, 0.7]과 [0.7, 0.3]으로 바꿔 비교해보세요  
2️⃣ HyDE + Reranking을 조합한 Retriever를 만들어보세요  
3️⃣ 자신의 문서로 각 기법의 검색 품질을 비교 평가해보세요  

---

## 📚 참고 자료
- [HyDE 논문](https://arxiv.org/abs/2212.10496)
- [Sentence Transformers Cross-Encoders](https://www.sbert.net/docs/cross_encoder/usage/usage.html)
- [LangChain Ensemble Retriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)
- [LangChain Parent Document Retriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)